In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
load_dotenv()
from product_rag_system import load_reviews, create_documents, create_vector_store, load_vector_store, query_product

/Users/wizardninja/Desktop/GA_Applied_AI_and_Deep_Learning/capstone/ProductReviewRAGSystem/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
df = pd.read_csv("Reviews.csv")

In [3]:
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")

Dataset shape: (568454, 10)

Columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']

Data types:
Id                         int64
ProductId                 object
UserId                    object
ProfileName               object
HelpfulnessNumerator       int64
HelpfulnessDenominator     int64
Score                      int64
Time                       int64
Summary                   object
Text                      object
dtype: object


In [4]:
def get_unique_top_products(df, n_products=5, max_overlap=0.5):
    product_counts = df.groupby('ProductId').size().sort_values(ascending=False)
    
    selected_products = []
    selected_reviews = set()
    
    for product_id in product_counts.index:
        if len(selected_products) >= n_products:
            break
            
        # Get this product's reviews
        product_reviews = set(df[df['ProductId'] == product_id]['Text'].tolist())
        
        # Check overlap with already selected reviews
        if selected_reviews:
            overlap = len(product_reviews & selected_reviews) / len(product_reviews)
            if overlap > max_overlap:
                print(f"Skipping {product_id}: {overlap:.0%} overlap with selected products")
                continue
        
        # Add this product
        selected_products.append(product_id)
        selected_reviews.update(product_reviews)
        print(f"Selected {product_id}: {len(product_reviews)} reviews")
    
    return selected_products

# Get unique top products
top_products = get_unique_top_products(df, n_products=5, max_overlap=0.5)

# Create subset
df_subset = df[df['ProductId'].isin(top_products)]

print(f"\nSubset size: {len(df_subset)}")
print(f"Number of products: {len(top_products)}")
print(f"\nReviews per product:")
print(df_subset.groupby('ProductId').size().sort_values(ascending=False))
print(f"\nRating distribution in subset:\n{df_subset['Score'].value_counts().sort_index()}")

# Save
df_subset.to_csv('reviews_subset.csv', index=False)
print("\nSaved to reviews_subset.csv")

Selected B007JFMH8M: 910 reviews
Selected B0026RQTGE: 630 reviews
Skipping B002QWHJOU: 100% overlap with selected products
Skipping B002QWP89S: 100% overlap with selected products
Skipping B002QWP8H0: 100% overlap with selected products
Selected B003B3OOPA: 623 reviews
Selected B001EO5Q64: 567 reviews
Selected B0026KPDG8: 562 reviews

Subset size: 3299
Number of products: 5

Reviews per product:
ProductId
B007JFMH8M    913
B0026RQTGE    632
B003B3OOPA    623
B001EO5Q64    567
B0026KPDG8    564
dtype: int64

Rating distribution in subset:
Score
1      76
2      64
3     163
4     514
5    2482
Name: count, dtype: int64

Saved to reviews_subset.csv


In [5]:
# only run this if vectorstore has not already been created
df = load_reviews('reviews_subset.csv')
documents = create_documents(df)
vectorstore = create_vector_store(documents)

In [8]:
results = query_product("What constructive feedback do customers have about this product", "B0026KPDG8")
print(results)

Customers have provided several constructive feedback points regarding the product, highlighting both positive and negative aspects:

1. **Flavor Variety and Customization**: Several customers expressed a desire for more control over flavor combinations. One reviewer suggested allowing customers to select their preferred flavors, indicating that personalization could enhance satisfaction.

2. **Texture Concerns**: There are mixed opinions about the texture of the chips. Some customers enjoyed the unique texture, describing it as different and eventually enjoyable, while others found it unsatisfactory and not crunchy enough. This suggests that improving the texture could appeal to a broader audience.

3. **Taste Consistency**: While many reviewers praised the flavors, some noted inconsistencies among different varieties. One customer specifically mentioned that certain flavors, like sweet potato, lacked taste compared to others. Ensuring a consistent flavor profile across all varieties 